[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/03_document_input/03_document_input_solutions.ipynb)

# 03. `document-input-example` 동행 — 연습 문제 해설

> 본문: [03_document_input.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/03_document_input/03_document_input.ipynb)

먼저 직접 풀어본 뒤에 보세요.

## 0. 환경 준비 — 프로젝트를 옆에 펼쳐두기

In [ ]:
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
print("Colab에서 실행 중:", IN_COLAB)

if IN_COLAB:
    # 이 노트북은 "예제 프로젝트를 옆에 두고 같이 읽는" 노트북입니다.
    # 그래서 설명만 하지 않고, 저장소를 통째로 내려받아 **실제 프로젝트 파일**을 열어봅니다.
    subprocess.run(["git", "clone", "-q", "https://github.com/karzit/temp.git", "/content/temp"], check=False)
    REPO_ROOT = "/content/temp"
    !pip install -q pydantic
else:
    # 로컬에서 열었다면 이 노트북 위치(notebooks/project-walkthrough/NN_xxx/)에서 3단계 위가 저장소 루트입니다.
    REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))

PROJECT = os.path.join(REPO_ROOT, "example-projects", "document-input-example")
SRC = os.path.join(PROJECT, "src")
print("프로젝트 경로:", PROJECT)
assert os.path.isdir(SRC), "프로젝트 경로를 찾지 못했습니다. 저장소 루트에서 노트북을 열었는지 확인하세요."

아래 `show()`는 이 노트북 전체에서 쓰는 도우미입니다. **설명 대신 진짜 프로젝트 파일을 그대로 출력**해서, 노트북과 코드가 어긋나지 않게 합니다.

In [ ]:
import re


def show(filename, start=None, end=None, grep=None):
    """프로젝트 파일의 실제 소스를 줄 번호와 함께 출력한다.

    설명을 읽는 것과 실제 코드를 보는 것 사이의 간격을 없애기 위한 도우미입니다.
    이 노트북에서 "코드 읽기"라고 나오는 곳은 전부 진짜 프로젝트 파일을 그대로 보여줍니다.

        show("crawl.py")                  전체
        show("crawl.py", 30, 45)          30~45번째 줄
        show("crawl.py", grep="def ")     'def '가 들어간 줄만
    """
    path = os.path.join(SRC, filename) if not os.path.isabs(filename) else filename
    lines = open(path, encoding="utf-8").read().splitlines()

    if grep:
        picked = [(i, l) for i, l in enumerate(lines, 1) if re.search(grep, l)]
    else:
        s = (start or 1) - 1
        e = end or len(lines)
        picked = [(i, l) for i, l in enumerate(lines[s:e], s + 1)]

    for i, line in picked:
        print(f"{i:>4} | {line}")


def show_file(relpath, **kwargs):
    """프로젝트 루트 기준 경로로 파일을 보여준다 (README, docker-compose 등)."""
    show(os.path.join(PROJECT, relpath), **kwargs)


# 프로젝트 소스를 import할 수 있도록 경로를 등록해둡니다.
if SRC not in sys.path:
    sys.path.insert(0, SRC)

In [ ]:
import json

os.environ.setdefault("OPENAI_API_KEY", "sk-dummy-not-used")

fake_key_path = os.path.abspath("fake-google-key.json")
with open(fake_key_path, "w", encoding="utf-8") as f:
    json.dump({"type": "service_account"}, f)
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = fake_key_path

ocr_text = """휴 가 신 청 서

성명: 김민수      부서: 개발l팀
신청구분: [v] 연차휴가  [ ] 경조휴가  [ ] 병가
기간: 2026 . 3 . 2 ~ 2026 . 3 . 6  (5일간)
사유: 가족 여행으로 인한 연차 사용입니다.
신청일: 2026-02-20
"""
print(ocr_text)

## 연습 1. 필드 추가하기 — OCR 오타를 견디는 `description` 쓰기

**문제**: `applicant_name`과 `department`를 추가하세요.
OCR 오타(`개발l팀`)가 있어도 잘 채워지려면 `description`을 어떻게 써야 할까요?

**핵심**: `description`은 LLM에게 주는 **지시문**입니다. 그러니 오타를 어떻게 다룰지도 여기 적습니다.

In [ ]:
from pydantic import BaseModel, Field


class RegulationInquiryV2(BaseModel):
    document_type: str = Field(description="서류의 종류 (예: 휴가신청서, 재직증명서, 초과근무신청서 등)")
    applicant_request: str = Field(description="신청자가 요청하는 핵심 내용을 한두 문장으로 요약")

    # ↓ 새로 추가한 두 필드. description을 왜 이렇게 썼는지가 중요하다.
    applicant_name: str = Field(
        default="",
        description=(
            "신청자 본인의 이름. '성명', '신청인', '작성자' 옆에 적힌 값을 찾으세요. "
            "결재란의 승인자 이름과 혼동하지 마세요. 없으면 빈 문자열."
        ),
    )
    department: str = Field(
        default="",
        description=(
            "신청자의 소속 부서명. OCR 오인식으로 숫자 1과 알파벳 l/I, 0과 O가 섞여 있을 수 있으니 "
            "한국어 부서명으로 자연스럽게 교정해서 적으세요 (예: '개발l팀' -> '개발1팀'). "
            "없으면 빈 문자열."
        ),
    )

    related_dates: list[str] = Field(default_factory=list, description="서류에 등장하는 날짜들 (YYYY-MM-DD, 없으면 빈 목록)")
    keywords: list[str] = Field(default_factory=list, description="규정 검색에 도움이 될 핵심 키워드 목록")


schema = RegulationInquiryV2.model_json_schema()
for name in ("applicant_name", "department"):
    print(f"--- {name}")
    print("   ", schema["properties"][name]["description"])

`description`에 넣은 세 가지를 보세요. 각각 다른 실패를 막습니다.

| 넣은 내용 | 막는 실패 |
|---|---|
| "'성명', '신청인' 옆의 값" | 어느 칸을 봐야 하는지 몰라 엉뚱한 값을 넣는 것 |
| "결재란 승인자와 혼동하지 마세요" | 서류에 이름이 여러 개일 때 팀장 이름을 가져오는 것 |
| "1과 l, 0과 O가 섞일 수 있으니 교정" | OCR 오타를 그대로 저장하는 것 |
| "없으면 빈 문자열" | 없는 값을 지어내는 것 |

**`default=""`를 준 것도 중요합니다.** 없으면 LLM이 필드를 빠뜨렸을 때 검증 에러가 나서
서류 전체 처리가 실패합니다. 이름을 못 읽었다고 나머지 정보까지 버릴 이유는 없습니다.

## 연습 2. 신뢰도 표시하기

**문제**: OCR이 흐릿할 때 확신이 없음을 알리려면? 그 값이 낮으면 앱은 어떻게 동작해야 할까요?

**함정이 하나 있습니다.** LLM에게 "확신도를 0~1로 매겨라"라고 하면 숫자를 주긴 합니다.
그런데 그 숫자는 **근거가 약합니다.** LLM은 자기가 얼마나 맞는지 잘 모릅니다.
그래서 "몇 점"보다 **"왜 불확실한지"**를 같이 받는 게 훨씬 쓸모 있습니다.

In [ ]:
from typing import Literal


class RegulationInquiryV3(BaseModel):
    document_type: str = Field(description="서류의 종류")
    applicant_request: str = Field(description="요청 핵심 내용 요약")

    confidence: Literal["high", "medium", "low"] = Field(
        default="medium",
        description=(
            "이 추출 결과를 얼마나 신뢰할 수 있는지. "
            "high = 필요한 항목이 모두 명확히 읽힘. "
            "medium = 일부 글자가 흐릿하거나 추론으로 채운 항목이 있음. "
            "low = 서류 종류조차 불확실하거나 글자가 거의 안 읽힘."
        ),
    )
    uncertain_fields: list[str] = Field(
        default_factory=list,
        description="확신이 낮은 필드 이름 목록 (예: ['department', 'related_dates']). 없으면 빈 목록.",
    )


# 숫자 대신 3단계로 둔 이유: LLM이 0.73과 0.81을 일관되게 구분하지 못하기 때문이다.
# 세 단계면 사람도 LLM도 기준을 공유할 수 있고, 화면에서 분기하기도 쉽다.
example = RegulationInquiryV3(
    document_type="휴가신청서",
    applicant_request="연차휴가 5일 사용 신청",
    confidence="medium",
    uncertain_fields=["department"],
)
print(example.model_dump_json(indent=2))

### 앱은 어떻게 동작해야 하나

**신뢰도가 낮다고 그냥 거부하면 안 됩니다.** 사용자는 왜 안 되는지 모른 채 막힙니다.
**사람이 고칠 수 있게** 만드는 게 맞습니다.

In [ ]:
def render_result(result: RegulationInquiryV3) -> None:
    """신뢰도에 따라 화면 동작을 나눈다 (Streamlit 의사코드)."""
    if result.confidence == "high":
        print("✅ 자동 처리 진행")
    elif result.confidence == "medium":
        print("⚠️  아래 항목을 확인해주세요 (수정 가능하게 입력창으로 표시):")
        for field in result.uncertain_fields:
            print(f"      - {field}: [사용자가 직접 수정]")
        print("   나머지 항목은 그대로 진행")
    else:
        print("❌ 사진이 흐릿해 읽지 못했습니다. 다시 촬영하거나 직접 입력해주세요.")
        print("   (읽어낸 부분만이라도 미리 채워서 보여주면 사용자 부담이 줄어듭니다)")


for level in ("high", "medium", "low"):
    print(f"--- confidence = {level}")
    render_result(RegulationInquiryV3(document_type="휴가신청서", applicant_request="연차 5일",
                                      confidence=level, uncertain_fields=["department"]))
    print()

## 연습 3. 검증 규칙 강화하기 — `field_validator`

**문제**: `related_dates`에 `"2026-13-45"`가 들어와도 지금은 통과합니다. 실제 날짜인지 검사하세요.

먼저 지금 상태를 확인합니다.

In [ ]:
class Loose(BaseModel):
    related_dates: list[str] = Field(default_factory=list)


print("검증 없이:", Loose(related_dates=["2026-13-45", "아무말"]).related_dates)
print("-> 13월 45일이 그대로 통과했습니다.")

In [ ]:
from datetime import date

from pydantic import field_validator


class RegulationInquiryV4(BaseModel):
    related_dates: list[str] = Field(default_factory=list, description="YYYY-MM-DD 형식 날짜 목록")

    @field_validator("related_dates")
    @classmethod
    def check_dates(cls, values: list[str]) -> list[str]:
        """실제로 존재하는 날짜만 남긴다.

        에러를 던지지 않고 걸러내는 쪽을 택한 이유: 날짜 하나가 이상하다고 서류 전체를
        버릴 이유가 없다. 다만 조용히 버리면 안 되므로 경고는 남긴다.
        (엄격하게 막아야 하는 값이라면 여기서 ValueError를 던지면 된다.)
        """
        valid = []
        for value in values:
            try:
                date.fromisoformat(value)  # 13월 45일 같은 값은 여기서 걸린다
                valid.append(value)
            except ValueError:
                print(f"    ⚠️  버림: {value!r} (실제 날짜가 아님)")
        return valid


print("검증 있음:")
result = RegulationInquiryV4(related_dates=["2026-03-02", "2026-13-45", "2026-02-30", "아무말"])
print("  남은 값:", result.related_dates)

`2026-02-30`도 걸렸습니다. 2월은 30일이 없으니까요.
정규식으로 `\d{4}-\d{2}-\d{2}` 형식만 검사했다면 이건 통과했을 겁니다.
**형식이 맞는 것과 실제로 존재하는 것은 다릅니다.**

### LLM을 믿지 않고 한 번 더 검사하는 게 왜 필요한가

정형 출력을 쓰면 **타입**은 보장됩니다. `list[str]`을 요구했으니 문자열 리스트가 옵니다.
하지만 **값의 의미**는 아무도 보장하지 않습니다.

| 보장되는 것 | 보장되지 않는 것 |
|---|---|
| `related_dates`가 리스트다 | 그 안이 실제 날짜다 |
| `department`가 문자열이다 | 실존하는 부서다 |
| 필드가 다 있다 | 값이 서류에 실제로 적혀 있던 것이다 |

LLM은 **그럴듯한 것**을 만드는 도구입니다. 그럴듯함과 정확함은 다릅니다.
`field_validator`는 그 사이의 마지막 관문입니다.

> 💡 04 노트북의 **조 번호 연속성 검증**도 정확히 같은 발상입니다.
> 도구가 준 결과를 그대로 믿지 않고, 도메인 규칙으로 한 번 더 확인하는 것.

## 정리

| 연습 | 핵심 |
|---|---|
| 필드 추가 | `description`은 주석이 아니라 지시문. 헷갈릴 지점을 미리 적어준다 |
| 신뢰도 | 점수보다 "왜 불확실한지". 거부 대신 **사람이 고칠 수 있게** |
| 검증 강화 | 정형 출력은 타입만 보장한다. 값의 의미는 직접 검사해야 한다 |

본문으로 돌아가기: [03_document_input.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/project-walkthrough/03_document_input/03_document_input.ipynb)